# Advanced Hybrid RAG for Arabic Poetry

Welcome to this production ready Retrieval Augmented Generation pipeline designed specifically for classical Arabic literature. 

This notebook utilizes the Arabic Poetry Dataset, processing over 1.6 million verses to build a comprehensive search and generation index. It tackles the unique challenges of Arabic text processing, such as archaic vocabulary, diacritics, and complex semantics, by implementing a robust, multi stage retrieval architecture. It concludes with automated LLM evaluation to measure system reliability.

**System Architecture and Pipeline:**

1. **Intent Classification:** Routing queries dynamically for Search, Meaning, Author, and Completion.
2. **Exact Match Bypass:** Fast lookup for high confidence literal phrase matching.
3. **Hybrid Retrieval:** Combining lexical sparse search using BM25 with dense semantic search using FAISS.
4. **Reciprocal Rank Fusion:** Mathematically merging disparate retrieval scores.
5. **Cross Encoder Reranking:** Precision focused reordering of the top candidate verses.
6. **Grounded Generation:** Strict prompting strategies to prevent LLM hallucinations.
7. **Automated Evaluation:** Integrated testing using DeepEval to measure Faithfulness and Answer Relevancy.

In [1]:
!pip install -q \
    faiss-cpu \
    sentence-transformers \
    rank-bm25 \
    openai \
    accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 79.4 MB/s eta 0:00:00


## Imports

Import the libraries required for data processing, retrieval, reranking, vector search, and generation.

In [2]:
import os
import re
import time
import pickle
import json

import faiss
import numpy as np
import pandas as pd

from collections import defaultdict

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

from kaggle_secrets import UserSecretsClient
from openai import OpenAI

## Load Dataset

Load the Arabic poetry dataset and inspect its columns.

The system will automatically try to detect:
- poem text
- author
- poem title

In [3]:
DATASET_PATH = (
    "/kaggle/input/datasets/"
    "mdanok/arabic-poetry-dataset/"
    "Arabic_Poetry_Dataset.csv"
)

df = pd.read_csv(DATASET_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nSample:")
display(df.head(3))

Dataset shape: (75022, 6)

Columns:
['poet_name', 'poet_era', 'poem_tags', 'poem_title', 'poem_text', 'poem_count']

Sample:


,poet_name,poet_era,poem_tags,poem_title,poem_text,poem_count
0,لقيط بن يعمر الإيادي,العصر الجاهلي,"قصائد هجاء, عموديه, بحر الوافر, قافية الدال (د)",سلام في الصحيفة من لقيط,سَلامٌ في الصَحيفَةِ مِن لَقيطٍ\nإِلى مَن بِال...,عدد الابيات : 4
1,لقيط بن يعمر الإيادي,العصر الجاهلي,"قصائد حزينه, عموديه, بحر البسيط, قافية الألف ...",يا دار عمرة من محتلها الجرعا,يا دارَ عَمرَةَ مِن مُحتَلِّها الجَرَعا\nهاجَت...,عدد الابيات : 60
2,لقيط بن يعمر الإيادي,العصر الجاهلي,"قصائد قصيره, عموديه, بحر الرجز, قافية الألف (ا)",وخاننا خوان في ارتباعنا,وَخانَنا خَوّانٌ في اِرتِباعِنا\nفَاِنفَدَّ لِ...,عدد الابيات : 1


## Detect Metadata Columns

The retrieval system needs metadata associated with every verse.

At minimum:
- verse
- author
- poem title

If the dataset does not contain an author column, the system will continue to work for verse retrieval and semantic questions, but it will not fabricate author information.

In [4]:
def find_column(columns, candidates):
    """
    Find a column using exact or partial matching.
    """

    normalized_columns = {
        str(col).strip().lower(): col
        for col in columns
    }

    # Exact match
    for candidate in candidates:
        key = candidate.strip().lower()

        if key in normalized_columns:
            return normalized_columns[key]

    # Partial match
    for col in columns:

        col_lower = str(col).strip().lower()

        for candidate in candidates:

            if candidate.strip().lower() in col_lower:
                return col

    return None


poem_col = find_column(
    df.columns,
    [
        "poem_text",
        "poem",
        "text",
        "poetry",
        "content"
    ]
)

author_col = find_column(
    df.columns,
    [
        "author",
        "author_name",
        "poet",
        "poet_name",
        "الشاعر",
        "اسم الشاعر"
    ]
)

title_col = find_column(
    df.columns,
    [
        "title",
        "poem_title",
        "poem_name",
        "اسم القصيدة",
        "عنوان القصيدة"
    ]
)

print("Detected columns")
print("----------------")
print("Poem   :", poem_col)
print("Author :", author_col)
print("Title  :", title_col)


if poem_col is None:
    raise ValueError(
        "Could not detect the poem text column. "
        "Please set poem_col manually."
    )

Detected columns
----------------
Poem   : poem_text
Author : poet_name
Title  : poem_title


## Arabic Text Normalization

Normalize Arabic text for lexical retrieval.

This removes:
- diacritics
- tatweel
- punctuation

And normalizes common Arabic character variants.

The original verse is always preserved.
Only the normalized version is used for retrieval.

In [5]:
ARABIC_DIACRITICS = re.compile(
    r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]"
)

ARABIC_PUNCTUATION = re.compile(
    r"""[!"#$%&'()*+,\-./:;<=>?@\[\]^_`{|}~،؛؟«»…]"""
)


def normalize_arabic(text):
    """
    Normalize Arabic text for lexical retrieval.
    """

    if text is None or pd.isna(text):
        return ""

    text = str(text)

    # Remove diacritics
    text = ARABIC_DIACRITICS.sub("", text)

    # Remove tatweel
    text = text.replace("ـ", "")

    # Normalize Alef variants
    text = re.sub(r"[إأآٱا]", "ا", text)

    # Normalize Yeh variants
    text = text.replace("ى", "ي")

    # Normalize Hamza forms
    text = text.replace("ؤ", "و")
    text = text.replace("ئ", "ي")

    # Normalize Taa Marbuta
    text = text.replace("ة", "ه")

    # Remove punctuation
    text = ARABIC_PUNCTUATION.sub(" ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

## Build Verse-Level Records

Each poem is split into independent verses while preserving metadata.

Every record contains:

- original verse
- normalized verse
- author
- poem title
- source row

In [6]:
records = []

for row_idx, row in df.iterrows():

    poem = row[poem_col]

    if pd.isna(poem):
        continue

    poem = str(poem)

    author = ""

    if author_col is not None:
        value = row[author_col]

        if not pd.isna(value):
            author = str(value).strip()

    title = ""

    if title_col is not None:
        value = row[title_col]

        if not pd.isna(value):
            title = str(value).strip()

    for verse in poem.split("\n"):

        verse = verse.strip()

        if len(verse.split()) < 3:
            continue

        normalized = normalize_arabic(verse)

        if not normalized:
            continue

        records.append({
            "verse": verse,
            "normalized": normalized,
            "author": author,
            "title": title,
            "source_row": int(row_idx)
        })


print(f"Total verse records: {len(records):,}")

Total verse records: 1,623,855


## Deduplicate Verses

Remove exact normalized duplicates while preserving the first associated metadata record.

In [7]:
unique_records = {}

for record in records:

    key = record["normalized"]

    if key not in unique_records:
        unique_records[key] = record


records = list(unique_records.values())

print(f"Unique verses: {len(records):,}")

Unique verses: 1,592,340


## Exact Search Index

Create a dictionary for O(1)-style lookup of normalized exact phrases.

This is especially important for poetry queries such as:

"حكم سيوفك في رقاب العذل"

where lexical matching is more reliable than semantic similarity.

In [8]:
exact_index = defaultdict(list)

for idx, record in enumerate(records):

    normalized = record["normalized"]

    exact_index[normalized].append(idx)


print(
    f"Exact index entries: {len(exact_index):,}"
)

Exact index entries: 1,592,340


## BM25 Sparse Retrieval

BM25 provides lexical retrieval and is useful when the user's query shares words with the poetry database.

In [9]:
print("Preparing BM25 corpus...")

tokenized_corpus = [
    record["normalized"].split()
    for record in records
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index ready.")

Preparing BM25 corpus...
BM25 index ready.


## Dense Embedding Model

Load a multilingual sentence-transformer model.

The dense retriever is used for semantic similarity rather than exact lexical matching.

In [10]:
EMBEDDING_MODEL_NAME = (
    "paraphrase-multilingual-MiniLM-L12-v2"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


## FAISS Dense Index

Use normalized embeddings with inner-product similarity.

For normalized vectors:

cosine similarity = inner product

The index is saved to Kaggle working storage so it can be reused.

In [11]:
FAISS_PATH = "/kaggle/working/arabic_poetry.faiss"
EMBEDDINGS_PATH = "/kaggle/working/arabic_poetry_embeddings.npy"


sentences = [
    record["verse"]
    for record in records
]


if (
    os.path.exists(FAISS_PATH)
    and os.path.exists(EMBEDDINGS_PATH)
):

    print("Loading existing FAISS index...")

    index = faiss.read_index(
        FAISS_PATH
    )

    sentence_embeddings = np.load(
        EMBEDDINGS_PATH
    ).astype("float32")

else:

    print(
        "Creating embeddings. "
        "This may take a while for 1.6M+ verses."
    )

    sentence_embeddings = embedding_model.encode(
        sentences,
        batch_size=256,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    sentence_embeddings = np.asarray(
        sentence_embeddings,
        dtype="float32"
    )

    d = sentence_embeddings.shape[1]

    index = faiss.IndexFlatIP(d)

    index.add(sentence_embeddings)

    np.save(
        EMBEDDINGS_PATH,
        sentence_embeddings
    )

    faiss.write_index(
        index,
        FAISS_PATH
    )


print("FAISS dimension:", index.d)
print("FAISS vectors:", index.ntotal)

Creating embeddings. This may take a while for 1.6M+ verses.


Batches:   0%|          | 0/6221 [00:00<?, ?it/s]

FAISS dimension: 384
FAISS vectors: 1592340


## Save Metadata

The FAISS index only stores vectors.

The metadata file maps each FAISS index position back to:
- verse
- author
- poem title

In [12]:
METADATA_PATH = "/kaggle/working/arabic_poetry_records.pkl"

with open(METADATA_PATH, "wb") as f:
    pickle.dump(records, f)

print("Metadata saved.")

Metadata saved.


## Exact Retrieval

Exact retrieval is checked first.

This is the highest-confidence retrieval method for literal poetry queries.

In [13]:
def exact_search(query, k=10):

    normalized_query = normalize_arabic(query)

    if not normalized_query:
        return []

    results = []

    # Exact normalized match
    if normalized_query in exact_index:

        for idx in exact_index[normalized_query][:k]:

            record = records[idx]

            results.append({
                "index": idx,
                "verse": record["verse"],
                "author": record["author"],
                "title": record["title"],
                "score": 1.0,
                "method": "exact"
            })

        return results

    return results

## BM25 Retrieval

Retrieve candidates using lexical similarity.

In [14]:
def bm25_search(query, k=30):

    normalized_query = normalize_arabic(query)

    tokens = normalized_query.split()

    if not tokens:
        return []

    scores = bm25.get_scores(tokens)

    top_indices = np.argsort(scores)[::-1][:k]

    results = []

    for idx in top_indices:

        score = float(scores[idx])

        if score <= 0:
            continue

        record = records[int(idx)]

        results.append({
            "index": int(idx),
            "verse": record["verse"],
            "author": record["author"],
            "title": record["title"],
            "score": score,
            "method": "bm25"
        })

    return results

## FAISS Retrieval

Retrieve semantic candidates using cosine similarity implemented as normalized inner-product search.

In [15]:
def semantic_search(query, k=30):

    query_embedding = embedding_model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    query_embedding = np.expand_dims(
        query_embedding,
        axis=0
    )

    start = time.time()

    scores, indices = index.search(
        query_embedding,
        k
    )

    elapsed_ms = (
        time.time() - start
    ) * 1000

    print(
        f"FAISS search time: {elapsed_ms:.2f} ms"
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        if idx < 0:
            continue

        record = records[int(idx)]

        results.append({
            "index": int(idx),
            "verse": record["verse"],
            "author": record["author"],
            "title": record["title"],
            "score": float(score),
            "method": "semantic"
        })

    return results

## Reciprocal Rank Fusion

Combine BM25 and FAISS rankings using Reciprocal Rank Fusion.

RRF avoids directly adding incompatible BM25 and vector similarity scores.

Formula:

RRF(d) = Σ 1 / (k + rank(d))

In [16]:
def reciprocal_rank_fusion(
    result_lists,
    rrf_k=60
):

    fused_scores = defaultdict(float)
    result_lookup = {}

    for results in result_lists:

        for rank, result in enumerate(
            results,
            start=1
        ):

            idx = result["index"]

            fused_scores[idx] += (
                1.0 / (rrf_k + rank)
            )

            result_lookup[idx] = result

    ranked_indices = sorted(
        fused_scores,
        key=fused_scores.get,
        reverse=True
    )

    fused_results = []

    for idx in ranked_indices:

        result = dict(
            result_lookup[idx]
        )

        result["rrf_score"] = (
            fused_scores[idx]
        )

        result["method"] = "hybrid"

        fused_results.append(result)

    return fused_results

## Cross-Encoder Reranker

The hybrid retriever generates candidates.

The reranker then evaluates each query-document pair jointly.

This second-stage ranking is used only on a small candidate set, not on the entire 1.6M corpus.

In [17]:
RERANKER_MODEL_NAME = (
    "BAAI/bge-reranker-v2-m3"
)

reranker = CrossEncoder(
    RERANKER_MODEL_NAME,
    max_length=512
)

print("Reranker loaded.")

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Reranker loaded.


## Rerank Hybrid Candidates

Rerank the top hybrid candidates and return the final documents.

In [18]:
def rerank_results(
    query,
    candidates,
    top_k=5
):

    if not candidates:
        return []

    pairs = [
        [
            query,
            candidate["verse"]
        ]
        for candidate in candidates
    ]

    scores = reranker.predict(
        pairs,
        show_progress_bar=False
    )

    reranked = []

    for candidate, score in zip(
        candidates,
        scores
    ):

        item = dict(candidate)

        item["reranker_score"] = float(
            score
        )

        reranked.append(item)

    reranked.sort(
        key=lambda x: x["reranker_score"],
        reverse=True
    )

    return reranked[:top_k]

## Hybrid Retriever

Retrieval strategy:

1. Exact match
2. BM25
3. FAISS
4. RRF fusion
5. Cross-encoder reranking

Exact matches are returned immediately because literal poetry matching has higher confidence than semantic retrieval.

In [19]:
def hybrid_retrieve(
    query,
    top_k=5,
    candidate_k=30
):

    # --------------------------------------------------------
    # 1. Exact search
    # --------------------------------------------------------

    exact_results = exact_search(
        query,
        k=top_k
    )

    if exact_results:

        print(
            f"Exact matches found: "
            f"{len(exact_results)}"
        )

        return exact_results

    # --------------------------------------------------------
    # 2. BM25
    # --------------------------------------------------------

    bm25_results = bm25_search(
        query,
        k=candidate_k
    )

    # --------------------------------------------------------
    # 3. FAISS
    # --------------------------------------------------------

    semantic_results = semantic_search(
        query,
        k=candidate_k
    )

    # --------------------------------------------------------
    # 4. RRF
    # --------------------------------------------------------

    fused_results = reciprocal_rank_fusion(
        [
            bm25_results,
            semantic_results
        ]
    )

    print(
        f"Hybrid candidates: "
        f"{len(fused_results)}"
    )

    # --------------------------------------------------------
    # 5. Reranking
    # --------------------------------------------------------

    final_results = rerank_results(
        query,
        fused_results[:candidate_k],
        top_k=top_k
    )

    return final_results

## Query Intent Classification

Route the query into one of four tasks:

- search
- meaning
- author
- completion

This prevents the LLM from treating every query as a request for explanation.

In [20]:
def classify_query(query):

    q = normalize_arabic(query)

    author_patterns = [
        "من قائل",
        "من الشاعر",
        "قائل البيت",
        "صاحب البيت",
        "لمن هذا البيت",
        "من صاحب هذا البيت"
    ]

    for pattern in author_patterns:

        if normalize_arabic(pattern) in q:
            return "author"

    meaning_patterns = [
        "ما معنى",
        "معنى البيت",
        "اشرح",
        "شرح البيت",
        "ماذا يعني",
        "ماذا يقصد"
    ]

    for pattern in meaning_patterns:

        if normalize_arabic(pattern) in q:
            return "meaning"

    completion_patterns = [
        "اكمل",
        "أكمل",
        "تكملة",
        "اكمل البيت",
        "أكمل البيت"
    ]

    for pattern in completion_patterns:

        if normalize_arabic(pattern) in q:
            return "completion"

    return "search"

## Query Cleaning

Remove common instructional prefixes before retrieval.

For example:

"Who said: حَكِّم سُيوفَكَ..."

becomes:

"حَكِّم سُيوفَكَ..."

In [21]:
def extract_search_text(query):

    q = query.strip()

    prefixes = [
        "من قائل:",
        "من قائل",
        "من الشاعر:",
        "من الشاعر",
        "قائل البيت:",
        "قائل البيت",
        "صاحب البيت:",
        "صاحب البيت",
        "ما معنى:",
        "ما معنى",
        "معنى البيت:",
        "معنى البيت",
        "اشرح:",
        "اشرح",
        "أكمل:",
        "أكمل",
        "اكمل:",
        "اكمل"
    ]

    for prefix in prefixes:

        if q.startswith(prefix):
            q = q[len(prefix):].strip()

    return q

## Context Construction

Convert retrieved documents into a structured context.

Metadata is included only when it exists in the dataset.

In [22]:
def build_context(results):

    blocks = []

    for i, result in enumerate(
        results,
        start=1
    ):

        block = []

        block.append(
            f"Result {i}"
        )

        block.append(
            f"Verse: {result['verse']}"
        )

        if result.get("author"):
            block.append(
                f"Author: {result['author']}"
            )

        if result.get("title"):
            block.append(
                f"Poem title: {result['title']}"
            )

        blocks.append(
            "\n".join(block)
        )

    return "\n\n".join(blocks)

## OpenAI Client

Load the API key from Kaggle Secrets.

Create the OpenAI client for grounded answer generation.

In [23]:
user_secrets = UserSecretsClient()

api_key = user_secrets.get_secret(
    "OPENAI_API_KEY"
)

client = OpenAI(
    api_key=api_key
)

## Grounded Generation

The LLM is responsible only for generating the final natural-language response.

It must not:
- invent verses
- invent authors
- use outside knowledge
- override retrieved metadata

The retrieval layer is the source of truth.

In [24]:
FALLBACK = (
    "Sorry, the requested information "
    "is not available in the current database."
)


def generate_answer(
    query,
    results,
    intent
):

    if not results:
        return FALLBACK

    context = build_context(results)

    # --------------------------------------------------------
    # SEARCH
    # --------------------------------------------------------

    if intent == "search":

        instructions = f"""
You are an Arabic poetry database assistant.

User query:
{query}

Retrieved database records:
{context}

Rules:

1. Answer only from the retrieved database records.

2. If the user is searching for a verse, phrase, or part of a verse,
   return the relevant retrieved verses only.

3. Do not explain the verses unless the user explicitly asks for an explanation.

4. Do not add commentary such as:
   "This phrase is repeated..."
   or
   "These verses mean..."

5. Preserve the retrieved verse text exactly.

6. Do not invent verses.

7. Do not use outside knowledge.

8. If the retrieved records do not answer the query, return exactly:
"{FALLBACK}"

Return only the answer.
"""

    # --------------------------------------------------------
    # MEANING
    # --------------------------------------------------------

    elif intent == "meaning":

        instructions = f"""
You are an Arabic poetry explanation assistant.

User query:
{query}

Retrieved database records:
{context}

Rules:

1. Explain the meaning of the requested verse in clear Arabic.

2. Base the explanation only on the retrieved text.

3. Do not invent historical context.

4. Do not invent an author.

5. Do not add information from outside knowledge.

6. If the retrieved context is insufficient, return exactly:
"{FALLBACK}"

Return a concise answer.
"""

    # --------------------------------------------------------
    # AUTHOR
    # --------------------------------------------------------

    elif intent == "author":

        instructions = f"""
You are an Arabic poetry database assistant.

User query:
{query}

Retrieved database records:
{context}

Rules:

1. Identify the author only from the explicit Author field in the retrieved records.

2. Never infer or guess the author from the verse.

3. Never use outside knowledge.

4. If an author is explicitly present, return:
"Author: [author name]"

5. If no author is explicitly present in the retrieved records, return exactly:
"{FALLBACK}"

6. Do not explain the verse unless explicitly requested.

Return only the answer.
"""

    # --------------------------------------------------------
    # COMPLETION
    # --------------------------------------------------------

    elif intent == "completion":

        instructions = f"""
You are an Arabic poetry database assistant.

User query:
{query}

Retrieved database records:
{context}

Rules:

1. Complete the requested verse only using the retrieved records.

2. Do not invent missing text.

3. Preserve the original retrieved wording and diacritics.

4. Do not use outside knowledge.

5. If the retrieved records do not contain the requested completion, return exactly:
"{FALLBACK}"

Return only the answer.
"""

    else:

        instructions = f"""
Answer the user's question only using the retrieved database records.

User query:
{query}

Retrieved records:
{context}

Do not use outside knowledge.
Do not invent information.

If the answer is not contained in the retrieved records, return exactly:
"{FALLBACK}"
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a grounded Arabic poetry "
                    "database assistant. "
                    "The retrieved context is the only "
                    "source of truth."
                )
            },
            {
                "role": "user",
                "content": instructions
            }
        ]
    )

    return (
        response
        .choices[0]
        .message
        .content
        .strip()
    )

## Full RAG Pipeline

The complete pipeline:

Query
→ intent detection
→ query cleaning
→ exact/BM25/FAISS retrieval
→ RRF
→ reranking
→ metadata
→ grounded LLM

In [25]:
def rag_pipeline(
    query,
    top_k=5,
    candidate_k=30,
    show_context=True
):

    print("\n" + "=" * 70)
    print("RAG PIPELINE")
    print("=" * 70)

    print("Original query:")
    print(query)

    # --------------------------------------------------------
    # 1. Intent
    # --------------------------------------------------------

    intent = classify_query(query)

    print("\nIntent:")
    print(intent)

    # --------------------------------------------------------
    # 2. Search query
    # --------------------------------------------------------

    search_query = extract_search_text(query)

    print("\nSearch query:")
    print(search_query)

    # --------------------------------------------------------
    # 3. Retrieval
    # --------------------------------------------------------

    start = time.time()

    results = hybrid_retrieve(
        search_query,
        top_k=top_k,
        candidate_k=candidate_k
    )

    retrieval_time = (
        time.time() - start
    ) * 1000

    print(
        f"\nTotal retrieval time: "
        f"{retrieval_time:.2f} ms"
    )

    print(
        f"Retrieved documents: "
        f"{len(results)}"
    )

    # --------------------------------------------------------
    # 4. Display context
    # --------------------------------------------------------

    if show_context:

        print("\n" + "-" * 70)
        print("RETRIEVED CONTEXT")
        print("-" * 70)

        for i, result in enumerate(
            results,
            start=1
        ):

            print(f"\n[{i}]")
            print(
                "Verse:",
                result["verse"]
            )

            print(
                "Author:",
                result.get("author", "")
            )

            print(
                "Title:",
                result.get("title", "")
            )

            print(
                "Method:",
                result.get("method", "")
            )

            if "rrf_score" in result:
                print(
                    "RRF:",
                    result["rrf_score"]
                )

            if "reranker_score" in result:
                print(
                    "Reranker:",
                    result["reranker_score"]
                )

    # --------------------------------------------------------
    # 5. Generate
    # --------------------------------------------------------

    answer = generate_answer(
        query,
        results,
        intent
    )

    print("\n" + "-" * 70)
    print("FINAL ANSWER")
    print("-" * 70)

    print(answer)

    return answer

## Test 1 — Exact Poetry Search

This test checks whether a literal phrase is retrieved correctly.

Expected behavior:
- exact retrieval
- no unnecessary explanation

In [26]:
rag_pipeline(
    "حكم سيوفك في رقاب العذل",
    top_k=5
)


RAG PIPELINE
Original query:
حكم سيوفك في رقاب العذل

Intent:
search

Search query:
حكم سيوفك في رقاب العذل
Exact matches found: 1

Total retrieval time: 0.04 ms
Retrieved documents: 1

----------------------------------------------------------------------
RETRIEVED CONTEXT
----------------------------------------------------------------------

[1]
Verse: حَكِّم سُيوفَكَ في رِقابِ العُذَّلِ
Author: عنترة بن شداد
Title: حكم سيوفك في رقاب العذل
Method: exact

----------------------------------------------------------------------
FINAL ANSWER
----------------------------------------------------------------------
حَكِّم سُيوفَكَ في رِقابِ العُذَّلِ


'حَكِّم سُيوفَكَ في رِقابِ العُذَّلِ'

## Test 2 — Meaning Question

The retriever finds the relevant verse and the LLM explains it.

The LLM must not introduce unsupported historical information.

In [27]:
rag_pipeline(
    "ما معنى حَكِّم سُيوفَكَ في رِقابِ العُذَّلِ؟",
    top_k=5
)


RAG PIPELINE
Original query:
ما معنى حَكِّم سُيوفَكَ في رِقابِ العُذَّلِ؟

Intent:
meaning

Search query:
حَكِّم سُيوفَكَ في رِقابِ العُذَّلِ؟
Exact matches found: 1

Total retrieval time: 0.05 ms
Retrieved documents: 1

----------------------------------------------------------------------
RETRIEVED CONTEXT
----------------------------------------------------------------------

[1]
Verse: حَكِّم سُيوفَكَ في رِقابِ العُذَّلِ
Author: عنترة بن شداد
Title: حكم سيوفك في رقاب العذل
Method: exact

----------------------------------------------------------------------
FINAL ANSWER
----------------------------------------------------------------------
البيت "حَكِّم سُيوفَكَ في رِقابِ العُذَّلِ" يعني أن الشخص يدعو إلى استخدام السيوف (القتال) ضد الذين يعيبون أو يلومون (العذل). يُظهر هذا البيت قوة وعزيمة في مواجهة الانتقادات.


'البيت "حَكِّم سُيوفَكَ في رِقابِ العُذَّلِ" يعني أن الشخص يدعو إلى استخدام السيوف (القتال) ضد الذين يعيبون أو يلومون (العذل). يُظهر هذا البيت قوة وعزيمة في مواجهة الانتقادات.'

## Test 3 — Author Question

The author is returned only when it is explicitly available in the retrieved metadata.

The system must never guess the author from general knowledge.

In [28]:
rag_pipeline(
    "من قائل: حَكِّم سُيوفَكَ في رِقابِ العُذَّلِ؟",
    top_k=5
)


RAG PIPELINE
Original query:
من قائل: حَكِّم سُيوفَكَ في رِقابِ العُذَّلِ؟

Intent:
author

Search query:
حَكِّم سُيوفَكَ في رِقابِ العُذَّلِ؟
Exact matches found: 1

Total retrieval time: 0.05 ms
Retrieved documents: 1

----------------------------------------------------------------------
RETRIEVED CONTEXT
----------------------------------------------------------------------

[1]
Verse: حَكِّم سُيوفَكَ في رِقابِ العُذَّلِ
Author: عنترة بن شداد
Title: حكم سيوفك في رقاب العذل
Method: exact

----------------------------------------------------------------------
FINAL ANSWER
----------------------------------------------------------------------
Author: عنترة بن شداد


'Author: عنترة بن شداد'

## Test 4 — Semantic Search

This query does not necessarily contain the exact wording of a verse.

FAISS helps retrieve semantically related poetry.

In [29]:
rag_pipeline(
    "أريد بيتاً عن الشجاعة ومواجهة الأعداء",
    top_k=5
)


RAG PIPELINE
Original query:
أريد بيتاً عن الشجاعة ومواجهة الأعداء

Intent:
search

Search query:
أريد بيتاً عن الشجاعة ومواجهة الأعداء
FAISS search time: 212.20 ms
Hybrid candidates: 59

Total retrieval time: 2619.72 ms
Retrieved documents: 5

----------------------------------------------------------------------
RETRIEVED CONTEXT
----------------------------------------------------------------------

[1]
Verse: وأكفاءُ الشجاعةِ والكفاحِ
Author: عبد الغفار الأخرس
Title: إذا نبت الديار بحر قوم
Method: hybrid
RRF: 0.027364110201042444
Reranker: 0.13399693369865417

[2]
Verse: شجاعٌ يَقمعُ الأعداءَ قَمْعَا
Author: محمد المعولي
Title: سقى بالمنحنى رسما وربعا
Method: hybrid
RRF: 0.015873015873015872
Reranker: 0.10379887372255325

[3]
Verse: عنِ الدّنْيا أُريدُ بها انْفِصَالا
Author: أبو العلاء المعري
Title: أعن وخد القلاص كشفت حالا
Method: hybrid
RRF: 0.015151515151515152
Reranker: 0.09179135411977768

[4]
Verse: شَجاً لِنَفسِ شُجاعِ الحَربِ مُعتَرِضاً
Author: الشريف الرضي
Title: جري النس

'1. وأكفاءُ الشجاعةِ والكفاحِ  \n   Author: عبد الغفار الأخرس  \n   Poem title: إذا نبت الديار بحر قوم  \n\n2. شجاعٌ يَقمعُ الأعداءَ قَمْعَا  \n   Author: محمد المعولي  \n   Poem title: سقى بالمنحنى رسما وربعا  \n\n3. شَجاً لِنَفسِ شُجاعِ الحَربِ مُعتَرِضاً  \n   Author: الشريف الرضي  \n   Poem title: جري النسيم على ماء العناقيد  \n\n4. شُجاعاً في الحُروبِ الثائِراتِ  \n   Author: عنترة بن شداد  \n   Poem title: إذا قنع الفتى بذميم عيش'

## Test 5 — Verse Completion

The system should only provide completion text that exists in the retrieved database.
It must never generate an invented continuation.

In [30]:
rag_pipeline(
    "أكمل: حَكِّم سُيوفَكَ",
    top_k=5
)


RAG PIPELINE
Original query:
أكمل: حَكِّم سُيوفَكَ

Intent:
completion

Search query:
حَكِّم سُيوفَكَ
FAISS search time: 194.11 ms
Hybrid candidates: 60

Total retrieval time: 1258.53 ms
Retrieved documents: 5

----------------------------------------------------------------------
RETRIEVED CONTEXT
----------------------------------------------------------------------

[1]
Verse: سُيوفُكَ وَالمُثَقَّفَةُ الدَوامي
Author: علي بن الجهم
Title: متى عطلت رباك من الخيام
Method: hybrid
RRF: 0.015151515151515152
Reranker: 0.3810215890407562

[2]
Verse: حَكِّم سُيوفَكَ في رِقابِ العُذَّلِ
Author: عنترة بن شداد
Title: حكم سيوفك في رقاب العذل
Method: hybrid
RRF: 0.016129032258064516
Reranker: 0.36431100964546204

[3]
Verse: بحدِّ سيوفك كالرّاجلِ
Author: الشريف المرتضى
Title: ألا ما لقلب بأيدي الغواة
Method: hybrid
RRF: 0.015384615384615385
Reranker: 0.23674055933952332

[4]
Verse: فَلَيتَ سُيوفَكَ في حاسِدٍ
Author: المتنبي
Title: فهمت الكتاب أبر الكتب
Method: hybrid
RRF: 0.0136986301369863
Reran

'حَكِّم سُيوفَكَ في رِقابِ العُذَّلِ'

## Test 6 — Unknown Query

The system should refuse when the retrieved evidence does not contain an answer.

In [31]:
rag_pipeline(
    "هذه عبارة غير موجودة تماماً في قاعدة البيانات",
    top_k=5
)


RAG PIPELINE
Original query:
هذه عبارة غير موجودة تماماً في قاعدة البيانات

Intent:
search

Search query:
هذه عبارة غير موجودة تماماً في قاعدة البيانات
FAISS search time: 194.45 ms
Hybrid candidates: 60

Total retrieval time: 2766.86 ms
Retrieved documents: 5

----------------------------------------------------------------------
RETRIEVED CONTEXT
----------------------------------------------------------------------

[1]
Verse: ولا بهذا الوجود قائمة
Author: عبد الغني النابلسي
Title: يا من غفلتم وجوهكم سود
Method: hybrid
RRF: 0.015873015873015872
Reranker: 0.2671697437763214

[2]
Verse: وهذه نسب ولا وجود لها
Author: محيي الدين بن عربي
Title: العين واحدة والأمر واحدة
Method: hybrid
RRF: 0.01639344262295082
Reranker: 0.09225738048553467

[3]
Verse: تفسيرها ليس يذكر
Author: عمارة اليمني
Title: قل للمشارف عني
Method: hybrid
RRF: 0.016129032258064516
Reranker: 0.05616960674524307

[4]
Verse: ليس فيه من موضع لسقام
Author: الصنوبري
Title: كيف أخفى من السقام وجسمي
Method: hybrid
RRF: 0.014705

'Sorry, the requested information is not available in the current database.'

## Inspect Retrieval Results

Inspect the retrieval layer independently from the LLM.

This is important because RAG quality should be evaluated at the retrieval level before evaluating generation.

In [32]:
test_queries = [
    "العزة",
    "الشجاعة",
    "الفخر",
    "الحب"
]

for query in test_queries:

    print("\n" + "=" * 80)
    print(query)
    print("=" * 80)

    results = hybrid_retrieve(
        query,
        top_k=5,
        candidate_k=30
    )

    for rank, result in enumerate(
        results,
        start=1
    ):

        print(
            f"{rank}. {result['verse']}"
        )


العزة
FAISS search time: 195.45 ms
Hybrid candidates: 60
1. إِكرامِ والمجدِ الرفيعِ
2. أَيَارَبِّ أَذَا الْعِزَّهْ
3. فَتَمَلُّها في العِزَّةِ ال
4. ويطلب العزَّة بابتذالها
5. تِ العزَّ والتقلُّلا

الشجاعة
FAISS search time: 197.12 ms
Hybrid candidates: 55
1. إِنَّ الشَجاعَةَ في العَزا
2. وأكفاءُ الشجاعةِ والكفاحِ
3. قَد سُوِّرَت بِشُجاعِ
4. شجاعةٌ سبَّبها جودُه
5. عَدَدَ الشَجاعَةِ وَالكَرَم

الفخر
FAISS search time: 192.50 ms
Hybrid candidates: 57
1. ظاهرة الفخرِ ومن باطنِها
2. حَسَدَ الفَخرُ مَفخَرَه
3. وَأَحَقَّهُم بِمَكارِمِ الفَخرِ
4. وَبِعَلْيَاهُ أوْجُهُ الفَخْرِ
5. ء والفخر لدى الفخر

الحب
FAISS search time: 193.38 ms
Hybrid candidates: 60
1. في الحب للمحبوب
2. وعلى الحب أعنّا
3. تَحَبَّب فَإِنَّ الحُبَّ اعِيَةُ الحُبِّ
4. إن ذل المحب حب العزيز
5. كبر الحب وقدماً


## Important

The FAISS index positions must always correspond to the same order as `records`.

If `records` changes, the old FAISS index must not be reused.

If the dataset processing changes, delete the old:
- FAISS index
- embeddings
- metadata

and rebuild them together.

In [33]:
print("Records:", len(records))
print("FAISS vectors:", index.ntotal)

assert len(records) == index.ntotal, (
    "Metadata and FAISS index are out of sync!"
)

print("Metadata and FAISS are synchronized.")

Records: 1592340
FAISS vectors: 1592340
Metadata and FAISS are synchronized.


## DeepEval Framework

Evaluate the RAG pipeline's reliability using DeepEval.
The evaluation tests for:
- **Faithfulness:** Ensures the LLM does not hallucinate and bases its answer strictly on the retrieved verses.
- **Answer Relevancy:** Ensures the generated answer directly addresses the user's query.

This cell dynamically uses the existing `hybrid_retrieve` and `generate_answer` functions without requiring any pipeline rebuilds.

In [34]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["CONFIDENT_API_KEY"] = user_secrets.get_secret("CONFIDENT_API_KEY")

print("Connection to the DeepEval dashboard is ready!")

Connection to the DeepEval dashboard is ready!


In [35]:
!pip install -q deepeval

from deepeval import evaluate
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
import os

os.environ["OPENAI_API_KEY"] = api_key

def evaluate_rag_pipeline(query):
    print("\n" + "=" * 70)
    print(f"EVALUATING: {query}")
    print("=" * 70)
    
    # 1. Process query using existing pipeline functions
    intent = classify_query(query)
    search_query = extract_search_text(query)
    
    # 2. Retrieve documents
    results = hybrid_retrieve(search_query, top_k=5, candidate_k=30)
    
    # Extract verses for the evaluation context
    retrieval_context = [res["verse"] for res in results]
    
    # 3. Generate answer
    actual_output = generate_answer(query, results, intent)
    
    print("\nGenerated Answer:")
    print(actual_output)
    print("-" * 70)
    print("Running DeepEval metrics...\n")
    
    # 4. Define evaluation metrics 
    faithfulness = FaithfulnessMetric(threshold=0.7, model="gpt-4o-mini")
    answer_relevancy = AnswerRelevancyMetric(threshold=0.7, model="gpt-4o-mini")
    
    # 5. Create test case
    test_case = LLMTestCase(
        input=query,
        actual_output=actual_output,
        retrieval_context=retrieval_context
    )
    
    # 6. Execute evaluation
    evaluate([test_case], [faithfulness, answer_relevancy])

# Run a test evaluation on a complex query
evaluate_rag_pipeline("اشرح معنى: حكم سيوفك في رقاب العذل وإذا نزلت بدار ذلّ فرحل")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.7/567.7 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 2.6 MB/s eta 0:00:00

EVALUATING: اشرح معنى: حكم سيوفك في رقاب العذل وإذا نزلت بدار ذلّ فرحل
FAISS search time: 196.65 ms
Hybrid candidates: 60

Generated Answer:
البيت "حكم سيوفك في رقاب العذل وإذا نزلت بدار ذلّ فرحل" للشاعر عنترة بن شداد يتحدث عن ضرورة استخدام القوة (السيوف) ضد من يعيب أو يلوم (العذل) الشخص. كما يشير إلى أنه إذا وجد الإنسان نفسه في مكان يفتقر إلى الكرامة والاحترام (دار ذل)، فعليه أن يترك هذا المكان ويبحث عن مكان آخر يليق به.
----------------------------------------------------------------------
Running DeepEval metrics...



✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score         ┃ Pass Rate                                     ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Faithfulness               │ 1.00                  │ 100.00% | passed=1 | failed=0                 │ 1         │
│  Answer Relevancy           │ 1.00                  │ 100.00% | passed=1 | failed=0                 │ 1         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=356756;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=259226;https://app.confident-ai.com/project/cmtehxthm001cql0tqbwir4w7/test-runs/cmtejz9tj0015ll0tddgzxifd/regression-testing\https://app.confident-ai.com/project/cmtehxthm001cql0tqbwir4w7/test-runs/cmtejz9tj0015ll0tddgzxifd/regression-testi]8;;\
]8;id=259226;https://app.confident-ai.com/project/cmtehxthm001cql0tqbwir4w7/test-runs/cmtejz9tj0015ll0tddgzxifd/regression-testing\ng]8;;\